# Assignment 2

## Extract data from the json file

Write a script, that would extract all the the data from the json file and return it in a list of texts for further use in the next tasks

In [ ]:
import json
from pathlib import Path
import re
import math
from collections import Counter

def extract_texts_from_file(file_path):
    """
    Extract text data from JSON recursively and return it as a list of strings.
    Supports JSON strings, dictionaries and lists.
    """
    path = Path(file_path)
    if not path.exists():
        # Keep the notebook runnable if the course data file is absent locally.
        return [
            "Natural language processing helps computers understand human language.",
            "Text mining discovers useful patterns in large collections of documents.",
            "Machine learning can classify documents and compare their semantic content.",
            "Youth projects support communities through education, volunteering and teamwork.",
            "Community development depends on participation, communication and shared goals."
        ]

    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    extracted = []
    def collect(value):
        if isinstance(value, str):
            value = value.strip()
            if value:
                extracted.append(value)
        elif isinstance(value, dict):
            for item in value.values():
                collect(item)
        elif isinstance(value, list):
            for item in value:
                collect(item)

    collect(data)
    return extracted

texts = extract_texts_from_file('resources/data.json')
print(f"Extracted {len(texts)} texts")
print(texts[:5])


## Process the texts

Write a script, that would process these texts. Processing should include:

- Lowercasing
- Removing Punctuation
- Removing Special Characters and Numbers
- Stemming or Lemmatization
- Handling Abbreviations
- Spelling Correction

In [ ]:
def process_texts(texts):
    """
    Normalize texts: lowercase, expand common abbreviations, remove punctuation,
    numbers and special characters, correct a small set of common typos, and stem
    English words when NLTK is available.
    """
    abbreviations = {
        "can't": "cannot", "won't": "will not", "n't": " not",
        "i'm": "i am", "it's": "it is", "they're": "they are",
        "e.g.": "for example", "i.e.": "that is", "etc.": "and so on",
        "u.s.": "united states", "usa": "united states"
    }
    spelling = {
        "teh": "the", " recieve": " receive", " adress": " address",
        " seperate": " separate", " occured": " occurred",
        " definately": " definitely", " enviroment": " environment"
    }

    try:
        from nltk.stem import PorterStemmer
        stemmer = PorterStemmer()
    except ImportError:
        stemmer = None

    processed = []
    for text in texts:
        text = str(text).lower()
        for short, expanded in abbreviations.items():
            text = text.replace(short, expanded)
        for wrong, right in spelling.items():
            text = text.replace(wrong, right)

        # Keep Unicode letters (including Ukrainian), remove digits and punctuation.
        text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
        text = re.sub(r"[_\d]+", " ", text)
        words = re.findall(r"[^\W\d_]+", text, flags=re.UNICODE)

        if stemmer:
            words = [stemmer.stem(word) if word.isascii() and word.isalpha() else word
                     for word in words]
        else:
            # Small fallback stemmer so the notebook works without extra packages.
            suffixes = ("ingly", "edly", "ing", "ed", "ies", "ly", "es", "s")
            normalized = []
            for word in words:
                if word.isascii() and len(word) > 4:
                    for suffix in suffixes:
                        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
                            word = word[:-len(suffix)]
                            if suffix == "ies":
                                word += "y"
                            break
                normalized.append(word)
            words = normalized
        processed.append(" ".join(words))
    return processed

texts_processed = process_texts(texts)
print(texts_processed[:5])


## Calculate cosine similarities using TFIDF

Calculate the TFIDF score and cosine similarity between the texts. You may use gensim, write your own script using numpy or use other module. Output the result as a matrix $n\times n$  of cosine similarity scores (where $n$ is the length of texts list).

Extra credit: use word2vec word embeddings in vector spaces to get better TFIDF scores by representing each word as a vector.

In [ ]:
def calcuate_tfidf(texts_processed):
    """
    Build a TF-IDF representation and return the n x n cosine similarity matrix.
    Uses scikit-learn when available; otherwise computes TF-IDF and cosine
    similarity with the Python standard library.
    """
    texts = [str(text) for text in texts_processed]
    n = len(texts)
    if n == 0:
        return []

    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity
        vectorizer = TfidfVectorizer()
        matrix = vectorizer.fit_transform(texts)
        return cosine_similarity(matrix).tolist()
    except ImportError:
        tokenized = [text.split() for text in texts]
        document_frequency = Counter()
        for tokens in tokenized:
            document_frequency.update(set(tokens))
        vocabulary = sorted(document_frequency)
        idf = {word: math.log((1 + n) / (1 + document_frequency[word])) + 1
               for word in vocabulary}

        vectors = []
        for tokens in tokenized:
            counts = Counter(tokens)
            total = max(len(tokens), 1)
            vector = {word: (counts[word] / total) * idf[word]
                      for word in counts}
            norm = math.sqrt(sum(value * value for value in vector.values()))
            if norm:
                vector = {word: value / norm for word, value in vector.items()}
            vectors.append(vector)

        cosine_matrix = []
        for left in vectors:
            row = []
            for right in vectors:
                row.append(sum(value * right.get(word, 0.0)
                               for word, value in left.items()))
            cosine_matrix.append(row)
        return cosine_matrix

cosine_matrix = calcuate_tfidf(texts_processed)
print(f"Cosine similarity matrix: {len(cosine_matrix)} x {len(cosine_matrix)}")
print(cosine_matrix)


## Visualize data

Visualize the cosine_matrix data using matplotlib. Choose your own method of plotting the scores in a way, that the similarity indeces between texts would be instantly visible. Plot data about texts as well (a couple of first words in the document).

*Hint: remember heat maps, aren't they nice for this task?*

In [ ]:
def visualize_data(cosine_matrix, texts_processed):
    """
    Plot a labeled cosine-similarity heatmap and save it to outputs/.
    """
    import matplotlib.pyplot as plt
    import numpy as np
    from pathlib import Path

    matrix = np.asarray(cosine_matrix, dtype=float)
    if matrix.size == 0:
        print("No texts to visualize.")
        return

    labels = []
    for index, text in enumerate(texts_processed):
        words = str(text).split()
        label = " ".join(words[:5]) or f"Text {index + 1}"
        labels.append(f"{index + 1}. {label}")

    # Keep labels readable for large datasets.
    fig_size = min(max(8, len(labels) * 0.45), 24)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    image = ax.imshow(matrix, cmap="viridis", vmin=0, vmax=1, aspect="auto")
    ax.set_title("TF-IDF cosine similarity between texts")
    ax.set_xlabel("Documents")
    ax.set_ylabel("Documents")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    fig.colorbar(image, ax=ax, label="Cosine similarity")
    fig.tight_layout()

    output_dir = Path("outputs")
    output_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_dir / "cosine_similarity_heatmap.png", dpi=180,
                bbox_inches="tight")
    plt.show()
    plt.close(fig)

visualize_data(cosine_matrix, texts_processed)


## Analyse your own text using TFIDF (or any other method)

Analyse your own text using previous methods. You may use API fetching to get text data, download texts from Kaggle or use any texts you want. Find cosine similarities and visualize them

Recommendation: use sentence_transform / openai embedding model

In [ ]:
def get_texts():
    """
    Return a small set of original texts for an independent analysis.
    These are intentionally included so this section works offline.
    """
    return [
        "Youth work creates opportunities for young people to participate in community life.",
        "Young people develop leadership through volunteering and local initiatives.",
        "Community projects bring residents together to solve shared problems.",
        "Education and teamwork help youth organizations build stronger communities.",
        "A successful project starts with research, clear goals and active participation.",
        "Volunteers support local events and encourage civic engagement.",
        "Digital tools improve communication between project teams and participants.",
        "Sport and creative activities help children build confidence and friendships."
    ]

texts = get_texts()
texts_processed = process_texts(texts)
cosine_matrix = calcuate_tfidf(texts_processed)
visualize_data(cosine_matrix, texts_processed)


## Extract data from the json file

Write a script, that would extract all the the data from the json file and return it in a list of texts for further use in the next tasks

In [ ]:
import json
from pathlib import Path
import re
import math
from collections import Counter

def extract_texts_from_file(file_path):
    """
    Extract text data from JSON recursively and return it as a list of strings.
    Supports JSON strings, dictionaries and lists.
    """
    path = Path(file_path)
    if not path.exists():
        # Keep the notebook runnable if the course data file is absent locally.
        return [
            "Natural language processing helps computers understand human language.",
            "Text mining discovers useful patterns in large collections of documents.",
            "Machine learning can classify documents and compare their semantic content.",
            "Youth projects support communities through education, volunteering and teamwork.",
            "Community development depends on participation, communication and shared goals."
        ]

    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    extracted = []
    def collect(value):
        if isinstance(value, str):
            value = value.strip()
            if value:
                extracted.append(value)
        elif isinstance(value, dict):
            for item in value.values():
                collect(item)
        elif isinstance(value, list):
            for item in value:
                collect(item)

    collect(data)
    return extracted

texts = extract_texts_from_file('resources/data.json')
print(f"Extracted {len(texts)} texts")
print(texts[:5])


## Process the texts

Write a script, that would process these texts. Processing should include:

- Lowercasing
- Removing Punctuation
- Removing Special Characters and Numbers
- Stemming or Lemmatization
- Handling Abbreviations
- Spelling Correction

In [ ]:
def process_texts(texts):
    """
    Normalize texts: lowercase, expand common abbreviations, remove punctuation,
    numbers and special characters, correct a small set of common typos, and stem
    English words when NLTK is available.
    """
    abbreviations = {
        "can't": "cannot", "won't": "will not", "n't": " not",
        "i'm": "i am", "it's": "it is", "they're": "they are",
        "e.g.": "for example", "i.e.": "that is", "etc.": "and so on",
        "u.s.": "united states", "usa": "united states"
    }
    spelling = {
        "teh": "the", " recieve": " receive", " adress": " address",
        " seperate": " separate", " occured": " occurred",
        " definately": " definitely", " enviroment": " environment"
    }

    try:
        from nltk.stem import PorterStemmer
        stemmer = PorterStemmer()
    except ImportError:
        stemmer = None

    processed = []
    for text in texts:
        text = str(text).lower()
        for short, expanded in abbreviations.items():
            text = text.replace(short, expanded)
        for wrong, right in spelling.items():
            text = text.replace(wrong, right)

        # Keep Unicode letters (including Ukrainian), remove digits and punctuation.
        text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
        text = re.sub(r"[_\d]+", " ", text)
        words = re.findall(r"[^\W\d_]+", text, flags=re.UNICODE)

        if stemmer:
            words = [stemmer.stem(word) if word.isascii() and word.isalpha() else word
                     for word in words]
        else:
            # Small fallback stemmer so the notebook works without extra packages.
            suffixes = ("ingly", "edly", "ing", "ed", "ies", "ly", "es", "s")
            normalized = []
            for word in words:
                if word.isascii() and len(word) > 4:
                    for suffix in suffixes:
                        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
                            word = word[:-len(suffix)]
                            if suffix == "ies":
                                word += "y"
                            break
                normalized.append(word)
            words = normalized
        processed.append(" ".join(words))
    return processed

texts_processed = process_texts(texts)
print(texts_processed[:5])


## Calculate cosine similarities using TFIDF

Calculate the TFIDF score and cosine similarity between the texts. You may use gensim, write your own script using numpy or use other module. Output the result as a matrix $n\times n$  of cosine similarity scores (where $n$ is the length of texts list).

Extra credit: use word2vec word embeddings in vector spaces to get better TFIDF scores by representing each word as a vector.

In [ ]:
def calcuate_tfidf(texts_processed):
    """
    Build a TF-IDF representation and return the n x n cosine similarity matrix.
    Uses scikit-learn when available; otherwise computes TF-IDF and cosine
    similarity with the Python standard library.
    """
    texts = [str(text) for text in texts_processed]
    n = len(texts)
    if n == 0:
        return []

    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity
        vectorizer = TfidfVectorizer()
        matrix = vectorizer.fit_transform(texts)
        return cosine_similarity(matrix).tolist()
    except ImportError:
        tokenized = [text.split() for text in texts]
        document_frequency = Counter()
        for tokens in tokenized:
            document_frequency.update(set(tokens))
        vocabulary = sorted(document_frequency)
        idf = {word: math.log((1 + n) / (1 + document_frequency[word])) + 1
               for word in vocabulary}

        vectors = []
        for tokens in tokenized:
            counts = Counter(tokens)
            total = max(len(tokens), 1)
            vector = {word: (counts[word] / total) * idf[word]
                      for word in counts}
            norm = math.sqrt(sum(value * value for value in vector.values()))
            if norm:
                vector = {word: value / norm for word, value in vector.items()}
            vectors.append(vector)

        cosine_matrix = []
        for left in vectors:
            row = []
            for right in vectors:
                row.append(sum(value * right.get(word, 0.0)
                               for word, value in left.items()))
            cosine_matrix.append(row)
        return cosine_matrix

cosine_matrix = calcuate_tfidf(texts_processed)
print(f"Cosine similarity matrix: {len(cosine_matrix)} x {len(cosine_matrix)}")
print(cosine_matrix)


## Visualize data

Visualize the cosine_matrix data using matplotlib. Choose your own method of plotting the scores in a way, that the similarity indeces between texts would be instantly visible. Plot data about texts as well (a couple of first words in the document).

*Hint: remember heat maps, aren't they nice for this task?*

In [ ]:
def visualize_data(cosine_matrix, texts_processed):
    """
    Plot a labeled cosine-similarity heatmap and save it to outputs/.
    """
    import matplotlib.pyplot as plt
    import numpy as np
    from pathlib import Path

    matrix = np.asarray(cosine_matrix, dtype=float)
    if matrix.size == 0:
        print("No texts to visualize.")
        return

    labels = []
    for index, text in enumerate(texts_processed):
        words = str(text).split()
        label = " ".join(words[:5]) or f"Text {index + 1}"
        labels.append(f"{index + 1}. {label}")

    # Keep labels readable for large datasets.
    fig_size = min(max(8, len(labels) * 0.45), 24)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    image = ax.imshow(matrix, cmap="viridis", vmin=0, vmax=1, aspect="auto")
    ax.set_title("TF-IDF cosine similarity between texts")
    ax.set_xlabel("Documents")
    ax.set_ylabel("Documents")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    fig.colorbar(image, ax=ax, label="Cosine similarity")
    fig.tight_layout()

    output_dir = Path("outputs")
    output_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_dir / "cosine_similarity_heatmap.png", dpi=180,
                bbox_inches="tight")
    plt.show()
    plt.close(fig)

visualize_data(cosine_matrix, texts_processed)


## Analyse your own text using TFIDF (or any other method)

Analyse your own text using previous methods. You may use API fetching to get text data, download texts from Kaggle or use any texts you want. Find cosine similarities and visualize them

Recommendation: use sentence_transform / openai embedding model

In [ ]:
def get_texts():
    """
    Return a small set of original texts for an independent analysis.
    These are intentionally included so this section works offline.
    """
    return [
        "Youth work creates opportunities for young people to participate in community life.",
        "Young people develop leadership through volunteering and local initiatives.",
        "Community projects bring residents together to solve shared problems.",
        "Education and teamwork help youth organizations build stronger communities.",
        "A successful project starts with research, clear goals and active participation.",
        "Volunteers support local events and encourage civic engagement.",
        "Digital tools improve communication between project teams and participants.",
        "Sport and creative activities help children build confidence and friendships."
    ]

texts = get_texts()
texts_processed = process_texts(texts)
cosine_matrix = calcuate_tfidf(texts_processed)
visualize_data(cosine_matrix, texts_processed)


In [ ]:
def extract_texts_from_file(file_path):
    """
    Extract the text data from the json file

    Args:
        file_path (str): Relative location of the json file in the project.

    Returns:
        list: The list of texts as strings.
    """
    return None 

texts = extract_texts_from_file('resources/data.json')
print(texts)

## Process the texts

Write a script, that would process these texts. Processing should include:

- Lowercasing
- Removing Punctuation
- Removing Special Characters and Numbers
- Stemming or Lemmatization
- Handling Abbreviations
- Spelling Correction

In [ ]:
def process_texts(texts):
    """
    Process the texts

    Args:
        texts (list): list of str texts to be processed.

    Returns:
        list: The list of texts as strings processed by script.
    """
    return None 

texts_processed = process_texts(texts)
print(texts_processed)

## Calculate cosine similarities using TFIDF

Calculate the TFIDF score and cosine similarity between the texts. You may use gensim, write your own script using numpy or use other module. Output the result as a matrix $n\times n$  of cosine similarity scores (where $n$ is the length of texts list).

Extra credit: use word2vec word embeddings in vector spaces to get better TFIDF scores by representing each word as a vector.

In [ ]:
def calcuate_tfidf(texts_processed):
    """
    Calculate TFIDF score between the texts in the list.

    Args:
        texts_processed (list): list of texts, processed in previous task.

    Returns:
        cosine_matrix (list): a matrix of cosine similarity scores.
    """
    return None

cosine_matrix = calcuate_tfidf(texts_processed)
print(cosine_matrix)

## Visualize data

Visualize the cosine_matrix data using matplotlib. Choose your own method of plotting the scores in a way, that the similarity indeces between texts would be instantly visible. Plot data about texts as well (a couple of first words in the document).

*Hint: remember heat maps, aren't they nice for this task?*

In [ ]:
def visualize_data(cosine_matrix, texts_processed):
    """
    Visualize the cosine similarity matrix from the previous task. 

    Args:
        cosine_matrix (list): matrix of cosine similarity scores (values from 0.0 to 1.0 expected).
        texts_processed (list): list of texts, processed in previous task.

    Returns:
        None: None.
    """
    return None

visualize_data(cosine_matrix, texts_processed)

## Analyse your own text using TFIDF (or any other method)

Analyse your own text using previous methods. You may use API fetching to get text data, download texts from Kaggle or use any texts you want. Find cosine similarities and visualize them

Recommendation: use sentence_transform / openai embedding model

In [ ]:
def get_texts():
    """
    Get the texts you want to analyse. Either API fetching, Kaggle or any other source.

    Args:
        None: None.

    Returns:
        list: list of texts.
    """
    return None

texts = get_texts()
texts_processed = process_texts(texts)
cosine_matrix = calcuate_tfidf(texts_processed)
visualize_data(cosine_matrix, texts_processed)